# Final Integrated Challenge


## Scenario

You are building a model that predicts whether a person's annual income exceeds **$50K**, using the **Adult Census Income** dataset from OpenML:

```python
from sklearn.datasets import fetch_openml

adult = fetch_openml(name="adult", version=2, as_frame=True)

Your task is to build a fully reproducible Scikit-learn training pipeline, track every experiment with
MLflow, and manage the project history with Git — bringing together everything from the last three
sessions into one complete workflow

# Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


import mlflow


from pathlib import Path


# Data Loading

In [2]:
adult = fetch_openml(name='adult', version=2, as_frame=True)

X = pd.DataFrame(data=adult.data)
y = pd.Series(adult.target, name=adult.target_names[0])

In [3]:
X.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States


In [4]:
def data_summary(df: pd.DataFrame, target_col: pd.Series) -> pd.DataFrame:
    """Prints basic data diagnostics and returns missing value summary.

    Args:
        df: Input DataFrame to analyze.
        target_col: Name of the target column to show class distribution for.

    Returns:
        DataFrame of columns with missing values, sorted by missing percentage.
    """
    df = df.copy()
    print('Data shape:')
    print(f'Number of rows: {df.shape[0]}, Number of columns: {df.shape[1]}')
    print("="* 100 )
    print("Target value distribution:")
    print(target_col.value_counts(normalize=True))
    print("="* 100 )
    missing_values = (df.isna().mean() * 100).round(2)
    col_types = df.dtypes
    unique_values = df.nunique()
    summary_df = pd.DataFrame({
        'perc_missing_values' : missing_values,
        'col_type' : col_types,
        'number_unique_values' : unique_values
    })
    return summary_df.sort_values(by='perc_missing_values', ascending=False)



data_summary(df=X, target_col=y)

Data shape:
Number of rows: 48842, Number of columns: 14
Target value distribution:
class
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


,perc_missing_values,col_type,number_unique_values
occupation,5.75,category,14
workclass,5.73,category,8
native-country,1.75,category,41
education,0.00,category,16
fnlwgt,0.00,int64,28523
age,0.00,int64,74
marital-status,0.00,category,7
education-num,0.00,int64,16
race,0.00,category,5
relationship,0.00,category,6


In [5]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
dtypes: category(8), int64(6)
memory usage: 2.6 MB


In [6]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
age,48842.0,38.643585,13.710510,17.0,28.0,37.0,48.0,90.0
fnlwgt,48842.0,189664.134597,105604.025423,12285.0,117550.5,178144.5,237642.0,1490400.0
education-num,48842.0,10.078089,2.570973,1.0,9.0,10.0,12.0,16.0
capital-gain,48842.0,1079.067626,7452.019058,0.0,0.0,0.0,0.0,99999.0
capital-loss,48842.0,87.502314,403.004552,0.0,0.0,0.0,0.0,4356.0
hours-per-week,48842.0,40.422382,12.391444,1.0,40.0,40.0,45.0,99.0


In [7]:
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(exclude=np.number).columns.tolist()

# Data Splitting

In [8]:
target_class = {'<=50K' : 0, '>50K' : 1}
y = y.map(target_class)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

len(X_train), len(y_train), len(X_test), len(y_test)

(39073, 39073, 9769, 9769)

In [9]:
def build_pipeline(num_cols: list = numerical_columns, cat_cols: list = categorical_columns) -> Pipeline:
    
    numerical_pipeline = Pipeline(steps=[
        ('impute', SimpleImputer(strategy="median")),
        ('scale', StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ('impute', SimpleImputer(strategy='constant', fill_value="missing")),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ])

    preprocess = ColumnTransformer(transformers=[
        ('num_preprocess', numerical_pipeline, numerical_columns),
        ('cat_preprocess', categorical_pipeline, categorical_columns)
    ])

    return Pipeline(steps=[
        ('preprocess', preprocess),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
    ])

In [ ]:
lr_pipeline = build_pipeline()


def model_evaluation(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return pd.DataFrame({
        'metric' : ['accuracy', 'precision', 'recall', 'f1-score'],
        'score' : [accuracy, precision, recall, f1]
    }), report

# Testing Full Pipeline
results, report = model_evaluation(model=lr_pipeline, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

In [11]:
results

,metric,score
0,accuracy,0.809090
1,precision,0.568372
2,recall,0.840890
3,f1-score,0.678282


In [27]:
print(report)

              precision    recall  f1-score   support

           0       0.94      0.80      0.86      7431
           1       0.57      0.84      0.68      2338

    accuracy                           0.81      9769
   macro avg       0.75      0.82      0.77      9769
weighted avg       0.85      0.81      0.82      9769



In [12]:
db_path = (Path("..") / "mlruns.db").resolve()
TRACKING_URI = f"sqlite:///{db_path}"

mlflow.set_tracking_uri(TRACKING_URI)
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///E:\Data Science\Python\IMT Course\Assignments\Income Classification\mlruns.db


In [13]:
EXPERIMENT_NAME = "Adult_Census_Pipeline"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)


Experiment name: Adult_Census_Pipeline
Experiment ID: 1


In [29]:
with mlflow.start_run(run_name="LogisticRegression_Baseline_Model") as run:
    mlflow.set_tags({
        "model_family" : "linear",
        "Model_rank" : "Baseline Model",
        "project_stage": "development",
        "dataset" : "adult_census",
        "developer" : "YA"
    })
    mlflow.log_params({f"model_{k}": v for k, v in lr_pipeline.named_steps['classifier'].get_params().items()})
    mlflow.log_metrics(dict(zip(results['metric'], results['score'])))

    report_path = Path("../artifacts/classification_report.txt")
    report_path.write_text(report, encoding="utf-8")
    mlflow.log_artifact(str(report_path), artifact_path="evaluation")
    mlflow.sklearn.log_model(sk_model=lr_pipeline, name='baseline_classifier', skops_trusted_types=["numpy.dtype"])

    print("Run ID:", run.info.run_id)
    print("Model logged successfully.")

2026/10/07 17:51:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Run ID: d02a7934e66d4ef98c7af88c43120391
Model logged successfully.
